##### Quick notebook to test out Prompt Evaluation
- Create a prompt
- Create a test dataset
- Create eval scripts to grade answers
- Iterate on prompt
- Repeat process

In [12]:
import json
import yaml
import logging
import os
import sys
from typing import Dict, List
from pathlib import Path
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:


sys.path.insert(0, str(Path(".").resolve()))



### Make sure API key is working, but not displayed

In [6]:
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("API Key loaded successfully.")
    print(f"API Key Length: {len(api_key)} characters")
    print(f"API Key: {api_key[:4]}")  
else:
    print("Failed to load Open API Key.")


API Key loaded successfully.
API Key Length: 164 characters
API Key: sk-p


### Connect to OpenAI

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),

)


In [47]:
from anthropic import Anthropic



In [13]:

def load_prompt(prompt_path: str) -> Dict:
    with open(prompt_path, 'r') as f:
        return yaml.safe_load(f)

def load_eval_set(eval_path: str) -> Dict:
    with open(eval_path, 'r') as f:
        return json.load(f)

In [22]:
prompt = load_prompt(f"{Path.cwd()}/prompts/fr_it_prompt_v1.yml")
eval_set = load_eval_set(f"{Path.cwd()}/evaluations/fr_it_prompt_eval_v1.json")


In [33]:
type(eval_set)

dict

In [36]:
for test_case in eval_set:
    # Format the user message
    user_msg = prompt['user_template']
    print(f"user_msg: {user_msg}")
    print(f"type: {type(user_msg)}")

user_msg: Translate the following english text:
{english_text}

type: <class 'str'>


In [23]:


def calculate_score(expected: Dict, actual: Dict, category: str) -> float:
    """
    Score logic:
    - Sentiment classification: exact match (0.6 weight)

    """
    sentiment_match = 1.0 if expected['output_context'] == actual['output_context'] else 0.0
    
    
    return (sentiment_match * 0.6) 

In [51]:
from difflib import SequenceMatcher
from typing import Dict
import re

def calculate_translation_similarity(expected: str, actual: str) -> float:
    """
    Compare two translations using multiple strategies.
    Returns a score between 0 and 1.
    """
    # Strategy 1: Exact match (best case)
    if expected.lower().strip() == actual.lower().strip():
        return 1.0
    
    # Strategy 2: Sequence matching (handles minor variations)
    sequence_ratio = SequenceMatcher(None, expected.lower(), actual.lower()).ratio()
    
    # Strategy 3: Word-level overlap (handles reordering)
    expected_words = set(re.findall(r'\b\w+\b', expected.lower()))
    actual_words = set(re.findall(r'\b\w+\b', actual.lower()))
    
    if not expected_words:  # Empty string edge case
        return 1.0 if not actual_words else 0.0
    
    word_overlap = len(expected_words & actual_words) / len(expected_words | actual_words)
    
    # Combine strategies: sequence matching is more strict, word overlap is more lenient
    combined_score = (sequence_ratio * 0.7) + (word_overlap * 0.3)
    
    return combined_score

def calculate_score(expected: Dict, actual: Dict, category: str) -> float:
    """
    Score logic:
    - Italian translation similarity: 0.3 weight
    - French translation similarity: 0.3 weight
    - Sentiment classification: exact match, 0.2 weight
    - Sentence type classification: exact match, 0.15 weight
    - Notes quality: presence check, 0.05 weight
    """
    
    scores = {}
    
    # 1. Italian translation comparison
    if 'italian' in expected and 'italian' in actual:
        italian_score = calculate_translation_similarity(
            expected['italian'],
            actual['italian']
        )
        scores['italian'] = italian_score
    else:
        scores['italian'] = 0.0
    
    # 2. French translation comparison
    if 'french' in expected and 'french' in actual:
        french_score = calculate_translation_similarity(
            expected['french'],
            actual['french']
        )
        scores['french'] = french_score
    else:
        scores['french'] = 0.0
    
    # 3. Sentiment classification (exact match)
    sentiment_match = 1.0 if (
        expected.get('sentiment', '').lower().strip() == 
        actual.get('sentiment', '').lower().strip()
    ) else 0.0
    scores['sentiment'] = sentiment_match
    
    # 4. Sentence type classification (exact match)
    sentence_type_match = 1.0 if (
        expected.get('sentence_type', '').lower().strip() == 
        actual.get('sentence_type', '').lower().strip()
    ) else 0.0
    scores['sentence_type'] = sentence_type_match
    
    # 5. Notes presence (0 or 1 - just checking if notes exist)
    notes_present = 1.0 if actual.get('notes', '').strip() else 0.0
    scores['notes'] = notes_present
    
    # Weighted average
    weights = {
        'italian': 0.30,
        'french': 0.30,
        'sentiment': 0.20,
        'sentence_type': 0.15,
        'notes': 0.05
    }
    
    total_score = sum(scores[key] * weights[key] for key in scores)
    
    return total_score, scores  # Return both total and breakdown

In [24]:
def parse_output(response_text: str) -> Dict:
    """
    Parse LLM response into structured output.
    Expected format: language: word with article
    Example: "French: le mot"
    """
    result = {
        "output_context": None,
        "fr": None,
        "it": None
    }
    
    lines = response_text.strip().split('\n')
    
    for line in lines:
        if 'French' in line or 'fr' in line.lower():
            # Extract French translation
            parts = line.split(':')
            if len(parts) > 1:
                result['fr'] = parts[1].strip()
        elif 'Italian' in line or 'it' in line.lower():
            # Extract Italian translation
            parts = line.split(':')
            if len(parts) > 1:
                result['it'] = parts[1].strip()
    
    # Set output_context (adjust based on your actual prompt structure)
    result['output_context'] = response_text.strip()
    
    return result

In [ ]:

def run_evaluation(llm_name: str, prompt_config: Dict, eval_set: Dict) -> List[Dict]:
    # client = client  # Use the global client instance
    if llm_name == "OpenAI":
        client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
        )
        model_name = "gpt-4o-mini"
    elif llm_name == "Anthropic":
        client = Anthropic(api_key=os.getenv("CLAUDE_COURSE_API_KEY"))
        model_name = "claude-sonnet-4-0"
    results = []
    
    for test_case in eval_set['translations']:
        # Format the user message
        english_text = test_case['english']  
        user_msg = prompt_config['user_template'].format(
            english_text=test_case['english']
        )
        
        # Call the LLM
    if llm_name == "OpenAI":
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=100,
            messages=[
            {"role": "system", "content": prompt_config['system']['role']},
            {"role": "user", "content": user_msg}
        ]
        )
        # Parse the response (OpenAI response structure is different)
        actual_output = parse_output(response.choices[0].message.content)
    elif llm_name == "Anthropic":
        response = client.messages.create(
            model=model_name,
            max_tokens=100,
            system=prompt_config['system']['role'],
            messages=[{"role": "user", "content": user_msg}]
        )
        # Score the response
        actual_output = parse_output(response.content[0].text)

        # score = calculate_score(
        #     test_case['expected_output'],
        #     actual_output,
        #     test_case['category']
        # )
        
        # results.append({
        #     "test_id": test_case['id'],
        #     "category": test_case['category'],
        #     "expected": test_case['expected_output'],
        #     "actual": actual_output,
        #     "score": score,
        #     "passed": score >= 0.8
        # })
        # print(f"Test ID: {test_case['id']}")
        # print(f"Expected: {test_case['expected_output']}")
        # print(f"Actual: {actual_output}")
        print(f"user_msg: {user_msg}")
        print(f"Actual: {actual_output}")
        # print(f"Score: {score}")

        # Prepare expected output (match your data structure)
        expected_output = {
            'italian': test_case['italian'],
            'french': test_case['french'],
            'sentiment': test_case['sentiment'],
            'sentence_type': test_case['type'],
            'notes': test_case.get('notes', '')
        }
        
        # Score with breakdown
        total_score, score_breakdown = calculate_score(
            expected=expected_output,
            actual=actual_output,
            category=test_case.get('category', 'general')
        )
        
        results.append({
            "english": english_text,
            "expected": expected_output,
            "actual": actual_output,
            "total_score": total_score,
            "score_breakdown": score_breakdown,
            "status": "success",
            "passed": total_score >= 0.75  # Adjust threshold as needed
        })
    
    return results
    # return results
run_evaluation("Anthropic", prompt, eval_set)

INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


user_msg: Translate the following english text:
Oversharing

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Condivisione eccessiva\n\n**French:** Surpartage\n\nNote: Both terms refer to the act of sharing too much personal information, typically in social contexts or on social media.', 'fr': '** Surpartage', 'it': '** Condivisione eccessiva'}


[{'english': 'Oversharing',
  'expected': {'italian': 'condividere troppo (informazioni personali)',
   'french': 'trop en dire / divulguer trop d’informations',
   'sentiment': 'negative/social',
   'sentence_type': 'non_translatable',
   'notes': 'Modern social concept requiring paraphrase rather than a single lexical item.'},
  'actual': {'output_context': 'Here are the translations:\n\n**Italian:** Condivisione eccessiva\n\n**French:** Surpartage\n\nNote: Both terms refer to the act of sharing too much personal information, typically in social contexts or on social media.',
   'fr': '** Surpartage',
   'it': '** Condivisione eccessiva'},
  'total_score': 0.0,
  'score_breakdown': {'italian': 0.0,
   'french': 0.0,
   'sentiment': 0.0,
   'sentence_type': 0.0,
   'notes': 0.0},
  'status': 'success',
  'passed': False}]